# Convolutional Neural Networks for Computer Vision
_Example of the use of CNNs for object detection_

---

Object detection is a task in computer vision that involves identifying the presence, location, and type of one or more objects in a given photograph. It is a challenging problem that involves building upon methods for object recognition (_e.g._, where are they), object localization (_e.g._, what are their extent), and object classification (_e.g._, what are they).

In recent years, deep learning techniques are achieving state-of-the-art results for object detection, such as on standard benchmark datasets and in computer vision competitions. Notable is the "You Only Look Once", or YOLO, a family of Convolutional Neural Networks that achieve near state-of-the-art results with a single end-to-end model that can perform object detection in real-time.

In this lab, we will first focus on the simpler problem of object localization. The localization problem considers that only one object is present on the image, while the detection problem tries to determine all the objects present on the image. Then, we will implement a simplified version of YOLO.

---

## Conventions used in this lab

This lab is written with `PyTorch`. Two conventions are used everywhere and are worth stating once and for all.

**1. Images are stored as `(N, C, H, W)`.** The channel axis comes *before* the spatial axes. The images are loaded with `numpy` in the `(N, H, W, C)` layout, which is the layout `matplotlib` expects for display, so the arrays are transposed at the point where they enter the network, and only there.

**2. The networks return _logits_, not probabilities.** No `sigmoid` and no `softmax` is applied inside the model. Both are applied explicitly, either inside the loss (`BCEWithLogitsLoss` and `cross_entropy` do it internally, in a numerically stable way) or at prediction time when we want probabilities to display. Keeping raw scores in the model is the standard `PyTorch` practice, and it will matter in Part II where the YOLO loss applies these functions itself.

## Setting up the environment

In [ ]:
import os, sys
import math
import PIL
from PIL import Image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

print("torch version:", torch.__version__)

In [ ]:
# All the tensors and models of this lab will be sent to this device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

---
# PART I: Object Localization

As said, in this section, we will focus on the simpler problem of locating a single object per image, _i.e._, locating the object associated with the dominant class of a classification algorithm.

As we have seen during the course, in both localization and detection, we try to simultaneously determine an object's position and class in the form of a bounding box. The bounding box is encoded by its width $b_w$, its height $b_h$, and its center whose coordinates are given by the pair $(b_x, b_y)$.

<center> <img src="img/Obama.png" style="width:500;height:300px;"></center>
<caption><center><b>Figure 1:</b> Bounding box model used for localization</center></caption>

## Data exploration

To begin, download the database images. _For the rest of the tutorial, we will need this data to be in a `wildlife` folder, in the current folder._

In [ ]:
!git clone https://plmlab.math.cnrs.fr/chevallier-teaching/datasets/wildlife.git

The database has 4 classes, for the following 4 animals: buffalo, elephant, rhinoceros and zebra. It contains **376** images of each of the 4 classes.

A fifth folder, `background`, holds images containing **no animal at all**. They were not photographed separately: each one is a window cut out of one of the images above, drawn at random and kept only if it intersects none of the annotated bounding boxes. Being cut out of the very same photographs, they share their lighting, their grain and their landscapes, and a network cannot tell them apart from the rest by any other means than the absence of an animal. The script that generated them, `make_background.py`, sits next to this notebook.

These images are of no use for Part I, which locates an animal known to be present; they will be used from Part II onwards. Two of their properties are worth knowing: their annotation file is empty, since there is nothing to annotate, and the filtering only knows about what was annotated in the first place, so a few crops still show a piece of an animal that the original files had missed.

<center> <img src="https://plmlab.math.cnrs.fr/chevallier-teaching/datasets/wildlife/-/raw/main/buffalo/002.jpg" width=200>
<img src="https://plmlab.math.cnrs.fr/chevallier-teaching/datasets/wildlife/-/raw/main/elephant/003.jpg" width=200>
<img src="https://plmlab.math.cnrs.fr/chevallier-teaching/datasets/wildlife/-/raw/main/rhino/003.jpg" width=200>
<img src="https://plmlab.math.cnrs.fr/chevallier-teaching/datasets/wildlife/-/raw/main/zebra/046.jpg" width=200></center>
<caption><center><b>Figure 2:</b> Examples of images from the database</center></caption>

##### <i style="color:teal">**Todo:** Display an image of the training dataset</i>

In [ ]:
### TO BE COMPLETED ###

# Display an image of the training dataset

In [ ]:
# %load solutions/VisionCNN/disp_img.py

##### <i style="color:teal">**Question:** What is the format used in the database to encode the labels?</i>

**[Solution]**

<!--
Each image `NNN.jpg` comes with a text file `NNN.txt`. Each line of that file describes one object, with five numbers separated by spaces: the class index, then the coordinates $b_x$, $b_y$ of the centre of the bounding box and its width $b_w$ and height $b_h$. The four coordinates are normalised by the size of the image, hence between $0$ and $1$, which makes them independent of the resizing we are about to apply.
-->

The function below will load the data and format it for classification. Some remarks about this function
* To be able to use our images in a reasonable amount of time, we will have to start by resizing them.
* We want the label to be of the form seen during the course, _i.e._, presence + bounding box + classes
* If there are several objects in the same image, we will consider only the object whose bounding box takes up the largest area in the image.

##### <i style="color:teal">**Question:** We consider square images, denoted $x$, of size $s$ and we want to store them in a `numpy` array.</i>
1. <span style="color:teal">How big should the array for the image $x$ be?</span>
2. <span style="color:teal">We also want to store the labels, denoted $y$, in a `numpy` array. What size should this array be?</span>

**[Solution]**

<!--
1. $x$ being a square image of size $s$, in color (3 channels), it is of size $(s,s,3)$.

2. $y$ is a vector that must contain "presence" + "bounding box" + "classes". It is therefore size $9 = 1+4+4$.

> Note that the images are loaded in the $(s,s,3)$ layout, which is the one `matplotlib` expects. The transposition to the $(3,s,s)$ layout expected by the network is done later, when the arrays are turned into tensors.
-->

The loading is split into two steps, and this separation is what the rest of the lab is built on.

**Reading** walks a folder and returns, for each image, the path to the file and the list of the objects it contains, exactly as they are written in the annotation files. Nothing is interpreted at this stage: an image with three animals keeps its three boxes, an image with none keeps an empty list.

**Encoding** turns that list into the label the network expects. This is where the choices of a given part are made: Part I keeps only the largest box and produces a vector of $9$; Part III will produce an $8 \times 8$ grid from the very same reading.

Two remarks on the reading function below. It pairs an image with its annotation **by name**, `007.jpg` with `007.txt`. And it records the photograph each sample comes from, which will be needed to split the base correctly.

In [ ]:
DATA_PATH = "wildlife"

FOLDERS_ANIMALS = [os.path.join(DATA_PATH, name)
                   for name in ["buffalo", "elephant", "rhino", "zebra"]]

In [ ]:
# Here we choose the dimension in which we will resize the images:
# 64x64 is a rather small size but it will allow us to have faster experiments
# 128x128 or 256x256 would give better results but at the cost of several hours of training
IMAGE_SIZE = 64

The classes are stored as a single index in the annotation files, whereas the label needs them as a "one-hot" vector. The following function performs that conversion.

In [ ]:
def one_hot(class_index, num_classes=4):
    """Turn a class index into a one-hot vector, e.g. 2 -> [0., 0., 1., 0.]."""
    return np.eye(num_classes, dtype="f")[int(class_index)]

one_hot(2)

In [ ]:
def read_annotations(folders):
    """Read the annotation files, without interpreting them.

    Returns a list of (path, boxes), one entry per image, where boxes is the list of the
    objects of that image as (class_index, cx, cy, width, height), normalized between 0 and 1.
    An image with no annotated object gets an empty list.
    """
    samples = []

    for folder in folders:
        for name in sorted(os.listdir(folder)):
            if not name.lower().endswith(".jpg"):
                continue

            stem = os.path.splitext(name)[0]
            annotation = os.path.join(folder, stem + ".txt")
            if not os.path.isfile(annotation):
                print("no annotation file for", name, "- image ignored")
                continue

            boxes = []
            with open(annotation) as f:
                for line in f:
                    fields = line.split()
                    if len(fields) < 5:
                        continue
                    boxes.append((int(fields[0]),) + tuple(float(v) for v in fields[1:5]))

            samples.append((os.path.join(folder, name), boxes))

    return samples


def load_images(samples, image_size=IMAGE_SIZE):
    """Load and resize the images of the samples, with color values brought back to [0, 1]."""
    x = np.zeros((len(samples), image_size, image_size, 3))

    for i, (path, _) in enumerate(samples):
        img = PIL.Image.open(path).convert("RGB")
        img = img.resize((image_size, image_size), PIL.Image.Resampling.LANCZOS)
        x[i] = np.asarray(img)

    return x / 255


def source_photograph(path):
    """Photograph a sample comes from. A background crop shares the source of its image."""
    folder = os.path.basename(os.path.dirname(path))
    stem = os.path.splitext(os.path.basename(path))[0]

    if folder == "background":
        # 'buffalo_007' or 'buffalo_007_1' -> 'buffalo/007'
        parts = stem.split("_")
        return parts[0] + "/" + parts[1]

    return folder + "/" + stem

##### <i style="color:teal">**Todo:** Complete the encoding function</i>

The label is the vector of size $9$ described above: presence, then the four coordinates, then the four class probabilities. If several objects are present, only the one whose bounding box takes up the largest area is kept.

A vector of zeros is returned when the image contains no object: presence $0$, and coordinates and classes left at zero since they mean nothing in that case. Part I never meets this situation, every animal photograph containing an animal, but Part II will.

In [ ]:
### TO BE COMPLETED ###

def encode_localization(boxes):
    """Turn the list of objects of an image into a label of size 9."""
    label = np.zeros(9, dtype="f")

    if len(boxes) == 0:
        # No object on this image: presence stays at 0, and so does the rest
        return label

    # Object whose bounding box takes up the largest area
    class_index, cx, cy, width, height = max(boxes, key=lambda box: ...) ### TO BE COMPLETED ###

    label[0] = ...   ### TO BE COMPLETED ###
    label[1:5] = ... ### TO BE COMPLETED ###
    label[5:] = ...  ### TO BE COMPLETED ###

    return label

In [ ]:
# %load solutions/VisionCNN/encode_localization.py

In [ ]:
samples = read_annotations(FOLDERS_ANIMALS)
print("images:", len(samples))
print("objects per image:", np.bincount([len(boxes) for _, boxes in samples]))

x = load_images(samples)
y = np.stack([encode_localization(boxes) for _, boxes in samples])
groups = np.array([source_photograph(path) for path, _ in samples])

x.shape, y.shape

##### <i style="color:teal">**Todo:** Prepare data for training</i>

The base is split into three parts: `train` to fit the parameters, `val` to compare the runs and choose when to stop, and `test`, left untouched until the very last cell of the lab. Selecting a model on a set and reporting its score on that same set gives a figure that is optimistic, by an amount nobody can estimate afterwards.

The split is performed by group rather than image by image. This has no effect in this part, where each group holds a single photograph, but it will matter as soon as the background crops come in: a crop and the image it was cut from must never end up on either side of the split, as they show the same animal in the same light.

It remains to center-reduce the coordinates of the bounding boxes, which is your job here. Careful with the statistics: they must describe the boxes, so they are computed on the training images that do contain an object, and applied to the three sets. The color values need nothing, `load_images` has already brought them back to $[0, 1]$.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit


def split_dataset(x, y, groups, val_size=0.15, test_size=0.15, seed=SEED):
    """Split into train / val / test, keeping the samples of a same photograph together."""
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    rest, test = next(splitter.split(x, y, groups))

    splitter = GroupShuffleSplit(n_splits=1, test_size=val_size / (1 - test_size), random_state=seed)
    train, val = next(splitter.split(x[rest], y[rest], groups[rest]))
    train, val = rest[train], rest[val]

    return (x[train], y[train]), (x[val], y[val]), (x[test], y[test])

In [ ]:
### TO BE COMPLETED ###

(x_train, y_train), (x_val, y_val), (x_test, y_test) = split_dataset(x, y, groups)

# Statistics of the boxes, computed on the training images containing an object
present = ...  ### TO BE COMPLETED ###
y_mean = ...   ### TO BE COMPLETED ###
y_std = ...    ### TO BE COMPLETED ###

# Center-reduce the coordinates of the three sets
for y_set in (y_train, y_val, y_test):
    y_set[:, 1:5] = ... ### TO BE COMPLETED ###

x_train.shape, x_val.shape, x_test.shape

In [ ]:
# %load solutions/VisionCNN/data_processing.py

> From now on, we will consider $\texttt{y\_mean}$ and $\texttt{y\_std}$, respectively the mean and standard deviation of the boxes of $\texttt{y\_train}$, as global variables.
>
> In particular, whenever we need to use these quantities within a function, we will be able to do so.

Some of these functions (the IoU coefficient below, for instance) receive tensors rather than `numpy` arrays. A `torch` copy of these two constants, placed on the same device as the models, is therefore prepared once and for all.

In [ ]:
y_mean_t = torch.tensor(y_mean, dtype=torch.float32, device=device)
y_std_t = torch.tensor(y_std, dtype=torch.float32, device=device)

The following function allows to visualize the objects located on the images, _i.e._ on each image the selected bounding box, in a suitable color, and its associated class.

##### <i style="color:teal">**Todo:** Write a function to visualize the data.</i>

1. If only $\texttt{x}$ and $\texttt{y}$ are indicated, an image number is randomly drawn and the $\texttt{y}$ label associated with the image is displayed
2. If a 2nd $\texttt{y}$, named $\texttt{y\_pred}$, is indicated, then the two labels are displayed side by side, so that they can be compared
3. Finally, you can also indicate the $\texttt{id}$ of the image you wish to view.

A bounding box color is assigned to each label.

> This function works on `numpy` arrays in the $(N, s, s, 3)$ layout, the one used at loading time.

In [ ]:
### TO BE COMPLETED ###

def draw_localization(img, lab, title, image_size=IMAGE_SIZE):
    """Display one image and, if it holds an object, its bounding box in the class color."""
    colors = ["royalblue", "limegreen", "purple", "darkorange"] # Different colors for different classes
    classes = ["Buffalo", "Elephant", "Rhino", "Zebra"]

    # Image display
    ... ### TO BE COMPLETED ###

    # Nothing else to draw when the label says that no object is present.
    # Every image does contain one in this part; this will change in Part II.
    if lab[0] <= 0.5:
        plt.title(title.format("no object"))
        return

    # Determining the class
    class_id = ... ### TO BE COMPLETED ###

    # Determining the coordinates of the bounding box in the image frame
    ax = (lab[1]*y_std[1] + y_mean[1]) * image_size
    ay = (lab[2]*y_std[2] + y_mean[2]) * image_size
    width = (lab[3]*y_std[3] + y_mean[3]) * image_size
    height = (lab[4]*y_std[4] + y_mean[4]) * image_size
    # Determining the extrema of the bounding box, namely the minimum and maximum x/y value
    p_x = [..., ...] ### TO BE COMPLETED ###
    p_y = [..., ...] ### TO BE COMPLETED ###
    # Display the bounding box in the right color
    [...] ### TO BE COMPLETED ###

    plt.title(title.format(classes[class_id]))

In [ ]:
# %load solutions/VisionCNN/draw_localization.py

The function below takes care of the rest: drawing an image at random when no `id` is given, and placing the ground truth and the prediction side by side when a second label is passed.

In [ ]:
def print_data_localization(x, y, y_pred=[], id=None, image_size=IMAGE_SIZE):
    if id is None:
        # Random drawing of an image in the database
        num_img = np.random.randint(x.shape[0]-1)
    else:
        num_img = id

    img = x[num_img]

    if np.any(y_pred):
        plt.subplot(1, 2, 1)

    draw_localization(img, y[num_img],
                      "Ground truth : Image " + str(num_img) + " - {}", image_size)

    if np.any(y_pred):
        plt.subplot(1, 2, 2)
        draw_localization(img, y_pred[num_img],
                          "Prediction: Image " + str(num_img) + " - {}", image_size)

In [ ]:
plt.figure(figsize=(15, 100))

for i in range(60): #x.shape[0]):
    plt.subplot(20, 3, i+1)
    print_data_localization(x_train, y_train, image_size=IMAGE_SIZE, id=i)

plt.show()

## Useful functions

1 . Computation of the **IoU** coefficient (Intersection over Union) between real and predicted bounding boxes.

##### <i style="color:teal">**Todo:** Complete the following function</i>

Let $\texttt{y\_pred\_coord}$ and $\texttt{y\_true\_coord}$ be the components corresponding to the bounding box coordinates for $\texttt{y\_pred}$ and $\texttt{y\_true}$ respectively.

> This function is called on the outputs of the network, so it receives `torch` tensors and must be written with `torch` operations: the elementwise operations `+`, `-`, `*`, `/` behave as in `numpy`, and `torch.maximum` / `torch.minimum` compare two tensors term by term.
>
> Note the order of the arguments: throughout the lab, the predictions come first and the ground truth second, as in every loss function of `PyTorch`.

In [ ]:
### TO BE COMPLETED ###

def compute_iou(y_pred_coord, y_true_coord):
    ### "Denormalization" of bounding box coordinates
    pred_box_xy = y_pred_coord[:, ...] * y_std_t[...] + y_mean_t[...] ### TO BE COMPLETED ###
    true_box_xy = y_true_coord[:, ...] * y_std_t[...] + y_mean_t[...] ### TO BE COMPLETED ###

    ### "Denormalization" of the width and height of bounding boxes
    pred_box_wh = y_pred_coord[:, ...] * y_std_t[...] + y_mean_t[...] ### TO BE COMPLETED ###
    true_box_wh = y_true_coord[:, ...] * y_std_t[...] + y_mean_t[...] ### TO BE COMPLETED ###

    # Computation of the minimum and maximum coordinates of the real bounding box
    true_mins = ... ### TO BE COMPLETED ###
    true_maxs = ... ### TO BE COMPLETED ###

    # Computation of the minimum and maximum coordinates of the predicted bounding box
    pred_mins = ... ### TO BE COMPLETED ###
    pred_maxs = ... ### TO BE COMPLETED ###

    # Determining the intersection of bounding boxes
    intersect_mins = torch.maximum(pred_mins, true_mins)
    intersect_maxs = torch.minimum(pred_maxs, true_maxs)
    intersect_wh = torch.clamp(intersect_maxs - intersect_mins, min=0.)
    intersect_areas = intersect_wh[:, 0] * intersect_wh[:, 1]

    # Area of predicted and actual bounding boxes
    true_areas = ... ### TO BE COMPLETED ###
    pred_areas = ... ### TO BE COMPLETED ###

    # Area of the union of predicted and real boxes
    union_areas = ... ### TO BE COMPLETED ###

    iou_scores = intersect_areas / union_areas
    return iou_scores

In [ ]:
# %load solutions/VisionCNN/compute_iou.py

`compute_iou` returns one score per image of the batch. The quantity we will follow during the training is its average over the batch.

In [ ]:
def iou_metric(y_pred_coord, y_true_coord):
    """Mean IoU over a batch, returned as a plain Python float."""
    return compute_iou(y_pred_coord, y_true_coord).mean().item()

2. Visualization of learning quality

The training loop we are about to write stores the value of each metric, epoch after epoch, in a dictionary named `history`: `history["coord_loss"]` is the list of the training values of the coordinate loss, and `history["val_coord_loss"]` the list of its validation values.

In [ ]:
def plot_training_analysis(history, metric='loss'):

    loss = history[metric]
    val_loss = history['val_' + metric]

    epochs = range(len(loss))

    plt.plot(epochs, loss, 'b', linestyle="--", label='Training ' + metric)
    plt.plot(epochs, val_loss, 'g', label='Validation ' + metric)
    plt.title('Training and validation ' + metric)
    plt.legend()

3. Batching of the data

A network is not fed with the whole array at once, but with batches of images. In `PyTorch`, this is the job of two objects:

* a `Dataset`, which knows how to return the $i$-th example of the base, and
* a `DataLoader`, which wraps a `Dataset` and takes care of grouping the examples into batches, of shuffling them at the beginning of each epoch (`shuffle=True`) and, if asked, of preparing them in parallel processes.

This is also the place where the images change layout, from $(N, s, s, 3)$ to $(N, 3, s, s)$.
The labels are kept as a single tensor of size $9$; they will be split into their three parts (presence, coordinates, classes) inside the training loop.

In [ ]:
def make_dataset(x, y):
    """Build a Dataset from the numpy arrays, transposing the images to the (N, C, H, W) layout."""
    x_t = torch.tensor(np.transpose(x, (0, 3, 1, 2)), dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32)
    return TensorDataset(x_t, y_t)


def make_loaders(x_train, y_train, x_val, y_val, batch_size):
    """Build the training and validation DataLoaders. Only the training one is shuffled."""
    train_loader = DataLoader(make_dataset(x_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(make_dataset(x_val, y_val), batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

## A first example of CNN for object localization

This part aims to develop a first example of a convolutional network for object localization.

To do this, complete the codes provided to obtain a localization algorithm. You can use any convolutional base of your choice, but you must pay special attention to the output layer. Actually, you will produce three different outputs:
* one characterizing the presence of an object,
* another providing the coordinates of the bounding box, and finally,
* a last one performing the classification.

<center> <img src="img/CNN.png" width=500></center>
<caption><center><b>Figure 3:</b> Illustration of the network architecture to be built.</center></caption>

A network is a class inheriting from `nn.Module`, with two methods:

* `__init__` declares the layers, _i.e._ everything that holds parameters. `nn.Sequential` chains layers that are applied one after the other, which is convenient for the convolutional base.
* `forward` describes the computation performed on a batch `x` of shape $(N, 3, s, s)$. Because our network has three heads, `forward` simply returns the three results as a tuple.

Two remarks on the layers used below:

* `nn.Conv2d(in_channels, out_channels, kernel_size, padding='same')` needs the number of *input* channels, which `nn.Sequential` does not infer for you; keeping track of it is part of the exercise.
* Activation functions are layers of their own (`nn.ReLU()`), placed after the convolution.

The three heads return raw scores. The `sigmoid` of the presence and the `softmax` of the classes are applied later, by the losses and by the prediction function.

The default initialization of `PyTorch` is a uniform He initialization. To reproduce the usual `he_normal`, the following function is applied to every layer of the model.

In [ ]:
def init_he_normal(module):
    """He (normal) initialization of the weights, zero initialization of the biases."""
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
        if module.bias is not None:
            nn.init.zeros_(module.bias)

##### <i style="color:teal">**Todo:** Complete the following class</i>

In [ ]:
### TO BE COMPLETED ###

class LocalizationNet(nn.Module):

    def __init__(self, image_size=IMAGE_SIZE):
        super().__init__()

        ### TO BE COMPLETED ###
        # Convolutions, Pooling, Dropout,... Up to you to form your network!
        self.features = nn.Sequential(
            nn.Conv2d(3, ..., 3, padding='same'),
            nn.ReLU(),
            ...
        )

        # Number of features seen by the three heads, once the output of self.features is flattened.
        # Each MaxPool2d(2) divides the spatial size by 2.
        n_features = ... ### TO BE COMPLETED ###

        self.head_p = nn.Linear(n_features, ...)       # Output characterizing the presence of an object
        self.head_coord = nn.Linear(n_features, ...)   # Output characterizing bounding box coordinates
        self.head_classes = nn.Linear(n_features, ...) # Output characterizing the class probabilities

        self.apply(init_he_normal)

    def forward(self, x):
        # Flatten everything but the batch axis: (N, C, H, W) -> (N, C*H*W)
        f = torch.flatten(self.features(x), 1)
        return ... ### TO BE COMPLETED ###

In [ ]:
# %load solutions/VisionCNN/create_model_localization.py

Printing a model displays the layers it declares; the number of parameters has to be counted explicitly. Passing a batch of the right shape through the network is the quickest way to check the output sizes.

In [ ]:
model = LocalizationNet().to(device)
print(model)
print("\nNumber of trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

# Shapes of the three outputs, on a fake batch of 2 images
with torch.no_grad():
    out_p, out_coord, out_classes = model(torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device))
print("\noutputs:", out_p.shape, out_coord.shape, out_classes.shape)

## Training

You must associate a cost function with each network output to train your network. The total cost function will be the sum of the three previously defined cost functions, weighted by weights defined in the variable `loss_weights`.

The losses are ordinary functions returning a scalar tensor. Three of them are available here:

* `F.binary_cross_entropy_with_logits(pred, target)` expects raw scores for `pred` and a target in $\{0, 1\}$ of the same shape;
* `F.mse_loss(pred, target)` is the mean square error;
* `F.cross_entropy(pred, target)` expects raw scores for `pred`; the target may be given either as class indices or, as here, as the one-hot vectors we built at loading time.

**Take the time to test different values of** `loss_weights` **depending on the evolution of the metrics you observe during the training**.

<br>

> Remark: on this database, every image contains an animal, so `y[:, 0]` is $1$ everywhere and the presence output has strictly nothing to learn. Its accuracy will sit at $1$ from the first epoch and its loss will collapse towards zero, whatever weight you give it. <br>
> Leave `loss_weights['p']` alone and adjust the other two; Part II will give this output something to do.

##### <i style="color:teal">**Todo:** Write the three losses</i>

In [ ]:
### TO BE COMPLETED ###

def presence_loss(pred, target, presence):
    """Loss of the presence output. `pred` holds raw scores, `target` is 0 or 1."""
    return ... ### TO BE COMPLETED ###


def coord_loss(pred, target, presence):
    """Loss of the coordinates, both already center-reduced."""
    return ... ### TO BE COMPLETED ###


def class_loss(pred, target, presence):
    """Loss of the classification. `pred` holds raw scores, `target` one-hot vectors."""
    return ... ### TO BE COMPLETED ###


losses = {'p': presence_loss, 'coord': coord_loss, 'classes': class_loss}

In [ ]:
# %load solutions/VisionCNN/losses_localization.py

The training loop itself is written below. Two functions are enough:

* `run_epoch` performs **one** pass over a `DataLoader`. It is used both for training and for evaluating: the only difference is that, in evaluation, the gradients are neither computed nor applied. The presence of an optimizer is what tells the two apart.
* `fit_localization` calls `run_epoch` once on the training set and once on the validation set, for each epoch, and stores every metric in the `history` dictionary.

Two points deserve attention.

`model.train()` and `model.eval()` do not train or evaluate anything: they switch the *mode* of the layers whose behaviour differs between the two, `nn.Dropout` here. Forgetting the switch is a classic source of validation scores that are worse than they should be.

The gradients accumulate in `PyTorch`: `optimizer.zero_grad()` resets them before each `backward()`. Omitting it silently sums the gradients of successive batches.

Finally, `.item()` extracts a plain Python number out of a one-element tensor. Accumulating the tensors themselves would keep the whole computational graph alive, and the memory with it.

##### <i style="color:teal">**Todo:** Complete the two following functions</i>

In [ ]:
### TO BE COMPLETED ###

METRIC_NAMES = ['loss', 'p_loss', 'coord_loss', 'classes_loss',
                'p_accuracy', 'coord_iou', 'classes_accuracy']

# Metrics that only make sense on the images containing an object
ON_PRESENT_ONLY = {'coord_loss', 'classes_loss', 'coord_iou', 'classes_accuracy'}


def run_epoch(model, loader, losses, loss_weights, optimizer=None):
    """One pass over `loader`. If `optimizer` is None, the pass is a plain evaluation.

    Returns a dictionary giving the average of each metric over the epoch.
    """
    is_train = optimizer is not None
    model.train(is_train)

    totals = {name: 0.0 for name in METRIC_NAMES}
    n_seen, n_present = 0, 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        # The three targets, in the same order as the three outputs of the network
        t_p, t_coord, t_classes = y_batch[:, 0:1], y_batch[:, 1:5], y_batch[:, 5:9]

        with torch.set_grad_enabled(is_train):
            # Forward pass
            out_p, out_coord, out_classes = ... ### TO BE COMPLETED ###

            # One loss per output. The presence is passed to all three: the last two
            # need it to leave out the images that contain no object
            l_p = losses['p'](out_p, t_p, t_p)
            l_coord = ... ### TO BE COMPLETED ###
            l_classes = ... ### TO BE COMPLETED ###

            # ... combined into the total loss
            loss = ... ### TO BE COMPLETED ###

        if is_train:
            ### TO BE COMPLETED ###
            # Reset the gradients, backpropagate, update the parameters (three lines)
            ...

        # Accumulation of the metrics, weighted by the number of images they were computed on
        with torch.no_grad():
            n_batch = x_batch.shape[0]
            n_seen += n_batch
            totals['loss'] += loss.item() * n_batch
            totals['p_loss'] += l_p.item() * n_batch
            # An object is predicted as present when its score is positive, i.e. sigmoid(score) > 0.5
            totals['p_accuracy'] += ((out_p > 0).float() == t_p).float().mean().item() * n_batch

            # The box and the class only mean something on the images that contain an object
            present = t_p[:, 0] > 0.5
            n_batch_present = int(present.sum())
            if n_batch_present > 0:
                n_present += n_batch_present
                totals['coord_loss'] += l_coord.item() * n_batch_present
                totals['classes_loss'] += l_classes.item() * n_batch_present
                totals['coord_iou'] += iou_metric(out_coord[present], t_coord[present]) * n_batch_present
                totals['classes_accuracy'] += (out_classes[present].argmax(1)
                                               == t_classes[present].argmax(1)).float().mean().item() * n_batch_present

    # Each metric is divided by the number of images it was accumulated over
    counts = {name: (n_present if name in ON_PRESENT_ONLY else n_seen) for name in METRIC_NAMES}
    return {name: totals[name] / max(counts[name], 1) for name in METRIC_NAMES}


def fit_localization(model, train_loader, val_loader, optimizer, losses, loss_weights, epochs):
    """Train `model` for `epochs` epochs and return the history of the metrics."""
    history = {name: [] for name in METRIC_NAMES}
    history.update({'val_' + name: [] for name in METRIC_NAMES})

    for epoch in range(epochs):
        train_metrics = ... ### TO BE COMPLETED ###
        val_metrics = ... ### TO BE COMPLETED ###

        for name in METRIC_NAMES:
            history[name].append(train_metrics[name])
            history['val_' + name].append(val_metrics[name])

        print(f"Epoch {epoch+1}/{epochs}"
              f" - loss: {train_metrics['loss']:.4f} - val_loss: {val_metrics['loss']:.4f}"
              f" - val_IoU: {val_metrics['coord_iou']:.3f}"
              f" - val_class_acc: {val_metrics['classes_accuracy']:.3f}")

    return history

In [ ]:
# %load solutions/VisionCNN/train_loop.py

In [ ]:
epochs = 30
batch_size = 15

model = LocalizationNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

train_loader, val_loader = make_loaders(x_train, y_train, x_val, y_val, batch_size)

loss_weights = {'p': 1, 'coord': 1, 'classes': 1} ### Vary these WEIGHTS to find an ad hoc COMBINATION ###

history = fit_localization(model, train_loader, val_loader, optimizer,
                           losses, loss_weights, epochs)

### Graphical validation

1. _Analysis of the results:_ Curves of the evolution of the loss function and of the IoU of the bounding boxes, as well as of the accuracy of the predicted classes

In [ ]:
def plot_results(history):
    plt.figure(figsize=(15, 10))

    plt.subplot(2, 3, 1) ; plot_training_analysis(history, metric='p_loss')
    plt.subplot(2, 3, 2) ; plot_training_analysis(history, metric='coord_loss')
    plt.subplot(2, 3, 3) ; plot_training_analysis(history, metric='classes_loss')

    plt.subplot(2, 3, 4) ; plot_training_analysis(history, metric='p_accuracy')
    plt.subplot(2, 3, 5) ; plot_training_analysis(history, metric='coord_iou')
    plt.subplot(2, 3, 6) ; plot_training_analysis(history, metric='classes_accuracy')

    plt.show()

In [ ]:
plot_results(history)

2. Prediction of validation data

The prediction function below gathers the three outputs into a single array of size $9$, in the layout expected by `print_data_localization`. This is where the raw scores of the network are turned back into probabilities.

Three details make it a prediction function rather than a training one: `model.eval()` switches the layers to evaluation mode, `torch.no_grad()` spares the computation of the gradients, and `.cpu()` brings the results back to the memory `numpy` can read.

In [ ]:
@torch.no_grad()
def predict_localization(model, x, batch_size=64):
    """Return an array (N, 9): presence probability, coordinates, class probabilities."""
    model.eval()
    x_t = torch.tensor(np.transpose(x, (0, 3, 1, 2)), dtype=torch.float32)

    predictions = []
    for x_batch in DataLoader(x_t, batch_size=batch_size):
        out_p, out_coord, out_classes = model(x_batch.to(device))
        predictions.append(torch.cat([torch.sigmoid(out_p),
                                      out_coord,
                                      torch.softmax(out_classes, dim=1)], dim=1).cpu())

    return torch.cat(predictions).numpy()

In [ ]:
y_pred = predict_localization(model, x_val)
y_pred.shape

3. Display of results on several images

In [ ]:
for i in [2, 3, 7, 15, 16, 18, 24, 25]:
    plt.figure(figsize=(10, 5))
    print_data_localization(x_val, y_val, y_pred=y_pred, id=i, image_size=IMAGE_SIZE)

plt.show()

## Improvement of the cost function

In practice, it is tricky to find a good combination of loss functions as you did on the previous cells. The cross-entropy and the mean square error give values that are too different from being combined effectively.

A variant, perhaps simpler to do work, is to use only the mean square error as the loss for all outputs. This variant is implemented in the YOLO algorithm, of which we will implement a variant in the second part of this tutorial. Test this solution below. As in the previous exercise, feel free to vary the weights of the different elements of the cost function.

<br>

> Careful: the network returns raw scores, whereas the targets of the presence and of the classes live in $[0, 1]$. Comparing the two with a mean square error only makes sense once the `sigmoid` and the `softmax` have been applied, which the two cross-entropies used above were doing internally. Here they have to be written explicitly.

##### <i style="color:teal">**Todo:** Complete the following cell</i>

In [ ]:
### TO BE COMPLETED ###

epochs = 30
batch_size = 15

model = LocalizationNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

train_loader, val_loader = make_loaders(x_train, y_train, x_val, y_val, batch_size)

### TO BE COMPLETED ###
# Same three outputs, but every loss is now a mean square error
def presence_loss_mse(pred, target, presence):
    return ... ### TO BE COMPLETED ###


def coord_loss_mse(pred, target, presence):
    return ... ### TO BE COMPLETED ###


def class_loss_mse(pred, target, presence):
    return ... ### TO BE COMPLETED ###


losses_mse = {'p': presence_loss_mse, 'coord': coord_loss_mse, 'classes': class_loss_mse}

loss_weights = {'p': ..., 'coord': ..., 'classes': ...} ### TO BE COMPLETED ###

history = fit_localization(model, train_loader, val_loader, optimizer,
                           losses_mse, loss_weights, epochs)

In [ ]:
# %load solutions/VisionCNN/train_model_localization_MSE.py

### Graphical validation

1. _Analysis of the results:_ Curves of the evolution of the loss function and of the IoU of the bounding boxes, as well as of the accuracy of the predicted classes

In [ ]:
plot_results(history)

2. Prediction of validation data

In [ ]:
y_pred = predict_localization(model, x_val)

3. Display of results on several images

In [ ]:
for i in [2, 3, 7, 15, 16, 18, 24, 25]:
    plt.figure(figsize=(10, 5))
    print_data_localization(x_val, y_val, y_pred=y_pred, id=i, image_size=IMAGE_SIZE)

plt.show()

### Some axes of improvement

Considering the small size of the database, the results are not bad! There is some confusion between some classes, but the predictions are often interesting.

However, there should still be strong overfitting at this stage. There are several possibilities to correct it:

* _Regularization_ by weight decay, passed to the optimizer: `torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)`. Note that the penalty then applies to every parameter of the model, biases included; restricting it to the weights requires declaring two parameter groups.
* _Data augmentation_: with a `Dataset` applying the transformations of [`torchvision.transforms.v2`](https://docs.pytorch.org/vision/stable/transforms.html).
* Use of _transfer learning_: Starting from a network trained on *ImageNet* (which contains many classes of animals), we would benefit from very general filters which would help to limit overfitting.

#### Data augmentation

Rotating an image is easy; rotating its bounding box along with it is the part that usually needs care. The `v2` transforms of `torchvision` do both at once: a transform called on several arguments applies the *same* random draw to all of them, and adapts its effect to the nature of each one. A rotation turns the image and recomputes the coordinates of the box, a change of contrast leaves the box untouched.

Two ingredients make this work.

`tv_tensors` are thin wrappers around a tensor that say what it represents: `tv_tensors.Image` for an image, `tv_tensors.BoundingBoxes` for boxes, which also carry their `format` and the `canvas_size` of the image they refer to. A plain tensor would be treated as ordinary data and left as is.

`v2.RandomApply` wraps one or several transforms and applies them with probability `p`, which is how a probability is attached to a transform that does not have one of its own.

Note that the boxes are expressed here in absolute pixels and in the `CXCYWH` format (centre, width, height), whereas the base stores them normalized between $0$ and $1$; the conversion, in both directions, is done in the `Dataset` below. Finally, `ClampBoundingBoxes` cuts back to the image the boxes that a rotation or a translation has pushed over the edge.

In [ ]:
import random

from torchvision.transforms import v2
from torchvision import tv_tensors

In [ ]:
# Gamma correction of an image whose values live in [0, 1]. Applied to the image only:
# the second argument of v2.Lambda restricts it to the tv_tensors of that type.
random_gamma = v2.Lambda(lambda img: img.clamp(0, 1) ** random.uniform(0.8, 1.2),
                         tv_tensors.Image)

AUGMENTATIONS_TRAIN = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomApply([v2.RandomAffine(degrees=45, translate=(0.0625, 0.0625), scale=(0.9, 1.1))], p=0.5),
    v2.RandomApply([v2.ColorJitter(brightness=0.2, contrast=0.2)], p=0.5),
    v2.RandomApply([random_gamma], p=0.5),
    v2.ClampBoundingBoxes(),
])

# No augmentation on the validation set: the Dataset below simply takes transform=None

The augmentation is performed by a `Dataset` of our own, which replaces the `TensorDataset` used so far. Only two methods are needed:

* `__len__`, the number of examples;
* `__getitem__(idx)`, which returns the `idx`-th example, *already augmented*: the random draw happens again at every call, so the network never sees exactly the same image twice.

Everything else is unchanged: the `DataLoader` groups these examples into batches and reshuffles them at each epoch, so nothing has to be done by hand at the end of an epoch. Note also that the augmentation must be applied to the training set only; the validation set goes through the same class with `transform=None`.

The example is returned as plain tensors, in the layout the network expects. The class works on a copy of the label: modifying `self.y` would corrupt the base, quietly and for good.

In [ ]:
class WildLifeDataset(Dataset):

    def __init__(self, x_set, y_set, transform=None):
        self.x, self.y = x_set, y_set
        self.transform = transform

    # Number of examples of the base
    def __len__(self):
        return len(self.x)

    # Selection and augmentation of one example
    def __getitem__(self, idx):
        # Work on a copy: the labels of the base must not be modified
        label = self.y[idx].copy()

        # (H, W, 3) numpy image -> (3, H, W) tensor, tagged as an image
        img = tv_tensors.Image(np.transpose(self.x[idx], (2, 0, 1)), dtype=torch.float32)

        # Denormalization of the bounding box, then conversion to absolute pixel coordinates
        box = (label[1:5] * y_std[1:5] + y_mean[1:5]) * IMAGE_SIZE
        box = tv_tensors.BoundingBoxes(torch.tensor([box], dtype=torch.float32),
                                       format="CXCYWH",
                                       canvas_size=(IMAGE_SIZE, IMAGE_SIZE))

        # The same random draw is applied to the image and to its bounding box
        if self.transform is not None:
            img, box = self.transform(img, box)

        # Back to coordinates normalized as in the rest of the lab
        new_box = box.numpy()[0] / IMAGE_SIZE
        label[1:5] = (new_box - y_mean[1:5]) / y_std[1:5]

        # Same layout as the TensorDataset used so far: (C, H, W) image, label of size 9
        return img.as_subclass(torch.Tensor), torch.tensor(label, dtype=torch.float32)

In [ ]:
# Instantiation of a Dataset and of the DataLoader that batches it
batch_size = 16

train_dataset = WildLifeDataset(x_train, y_train, transform=AUGMENTATIONS_TRAIN)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# To test the dataset, we select the elements of the first batch and display them
batch_x, batch_y = next(iter(train_loader))

# Back to the numpy layout used by the display function
batch_x = batch_x.permute(0, 2, 3, 1).numpy()
batch_y = batch_y.numpy()

# --- #

plt.figure(figsize=(15, 25))

for i in range(15):
    plt.subplot(5, 3, i+1)
    print_data_localization(batch_x, batch_y, image_size=IMAGE_SIZE, id=i)

plt.show()

In [ ]:
model = LocalizationNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
batch_size = 18
epochs = 50

loss_weights = {'p': 1, 'coord': 5, 'classes': 1}

train_loader = DataLoader(WildLifeDataset(x_train, y_train, transform=AUGMENTATIONS_TRAIN),
                          batch_size=batch_size, shuffle=True)
val_loader = DataLoader(WildLifeDataset(x_val, y_val, transform=None),
                        batch_size=batch_size, shuffle=False)

history = fit_localization(model, train_loader, val_loader, optimizer,
                           losses, loss_weights, epochs)

1. _Analysis of the results:_ Curves of the evolution of the loss function and of the IoU of the bounding boxes, as well as of the accuracy of the predicted classes

In [ ]:
plot_results(history)

2. Prediction of validation data

In [ ]:
y_pred = predict_localization(model, x_val)

3. Display of results on several images

In [ ]:
for i in [2, 3, 7, 25, 16, 18, 24, 15]:
    plt.figure(figsize=(10, 5))
    print_data_localization(x_val, y_val, y_pred=y_pred, id=i, image_size=IMAGE_SIZE)

plt.show()

#### Transfer learning

`torchvision` gives access to the classical architectures together with their weights trained on *ImageNet*. The `.features` attribute of `VGG-16` is exactly its convolutional part, which is the part we want to reuse.

One precaution: these weights were obtained on images normalized channel by channel with the statistics of *ImageNet*, whereas ours are simply divided by $255$. The same normalization must therefore be applied before the convolutional base, otherwise the pretrained filters are fed with data they were never trained on. Below, it is done inside the model, so that the images stored in `x_train` stay directly displayable.

In [ ]:
from torchvision.models import vgg16, VGG16_Weights


def make_conv_base():
    """A fresh copy of the convolutional part of VGG-16, with its ImageNet weights.

    A function and not a global variable: the fine-tuning below modifies the weights it
    is given and unfreezes them. Reusing the same object from one experiment to the next
    would silently start the second one from the network trained by the first.
    """
    return vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features # The Dense part is not kept


make_conv_base()

##### <i style="color:teal">**Todo:** Write a class to create a neural network based on the $\texttt{VGG-16}$ convolution basis.</i>

In [ ]:
### TO BE COMPLETED ###

class LocalizationNetVGG(nn.Module):

    def __init__(self, conv_base, image_size=IMAGE_SIZE):
        super().__init__()
        self.conv_base = conv_base

        # Statistics of ImageNet, stored as buffers: they follow the model on the GPU,
        # but they are constants and not parameters to be learnt
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        # VGG-16 has five pooling layers and ends with 512 channels
        n_features = ... ### TO BE COMPLETED ###

        ### TO BE COMPLETED ###
        # The three heads, as in LocalizationNet
        [...]

    def forward(self, x):
        x = (x - self.mean) / self.std
        f = torch.flatten(self.conv_base(x), 1)
        return ... ### TO BE COMPLETED ###

In [ ]:
# %load solutions/VisionCNN/create_model_localization_VGG.py

In [ ]:
model = LocalizationNetVGG(make_conv_base()).to(device)
print("Total parameters:      ", sum(p.numel() for p in model.parameters()))
print("Trainable parameters:  ", sum(p.numel() for p in model.parameters() if p.requires_grad))

##### <i style="color:teal">**Todo:** Achieve Transfer Learning.</i>

Freezing a part of a network amounts to setting `requires_grad = False` on its parameters: the gradient is no longer computed for them, and the optimizer no longer updates them.

Two points to be careful about:

* the optimizer must be created **after** the freezing, and created **again** after the unfreezing, since it keeps a reference to the list of parameters it is in charge of;
* the fine-tuning is done with a much smaller learning rate, so as not to destroy the pretrained filters in the first few batches.

In [ ]:
### TO BE COMPLETED ###

model = LocalizationNetVGG(make_conv_base()).to(device)
batch_size = 18
epochs = 30

loss_weights = {'p': 1, 'coord': 5, 'classes': 1}

train_loader, val_loader = make_loaders(x_train, y_train, x_val, y_val, batch_size)

# --- #

print("Transfer learning")
### TO BE COMPLETED ###
# Freeze the convolutional base, then train the heads only
[...]

# --- #

print("\nFine tuning")
### TO BE COMPLETED ###
# Unfreeze everything and train again, with a much smaller learning rate
[...]

In [ ]:
# %load solutions/VisionCNN/transfert_learning.py

1. _Analysis of the results:_ Curves of the evolution of the loss function and of the IoU of the bounding boxes, as well as of the accuracy of the predicted classes

In [ ]:
plot_results(history)

2. Prediction on validation data

In [ ]:
y_pred = predict_localization(model, x_val)

3. Results display on several images

In [ ]:
for i in [2, 3, 7, 25, 16, 18, 24, 15]:
    plt.figure(figsize=(10, 5))
    print_data_localization(x_val, y_val, y_pred=y_pred, id=i, image_size=IMAGE_SIZE)

plt.show()

#### Data augmentation & Transfert learning

##### <i style="color:teal">**Todo:** Combine Transfer Learning and Data Augmentation.</i>

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/VisionCNN/transfert_learning_data_augment.py

In [ ]:
plot_results(history)

2. Prediction on validation data

In [ ]:
y_pred = predict_localization(model, x_val)

3. Results display on several images

In [ ]:
for i in [2, 3, 7, 25, 16, 18, 24, 15]:
    plt.figure(figsize=(10, 5))
    print_data_localization(x_val, y_val, y_pred=y_pred, id=i, image_size=IMAGE_SIZE)

plt.show()

---
# PART II: Is there anything to locate?

Part I rested on an assumption that was never stated: every image contains an animal. The presence output was therefore useless, its target being $1$ everywhere.

We now drop that assumption, and this changes the problem in a way that goes beyond adding a binary classification. When an image holds no object, its coordinates and its class have **no target**: there is no box to regress, and no species to name. The cost function can no longer be the same sum of three terms for every image; some of its terms must be switched off, image by image, depending on what the ground truth contains.

This conditional structure is the whole point of this part, and it is exactly the one you will find again in Part III, where a cell of the YOLO grid is empty or occupied in the same way that an image is here.

## Data

We reuse the reading and the encoding of Part I, adding the `background` folder to the list. Those images have an empty annotation file, so `read_annotations` gives them an empty list of objects, and `encode_localization` turns it into a label whose presence is $0$.

The split is the one written earlier, and this is where its grouping earns its keep: a background crop and the photograph it was cut from show the same animal in the same light, and must not be separated. `source_photograph` gives them the same group, so `split_dataset` keeps them together.

One detail on the standardization. The coordinates of an image without object are zeros, which mean nothing; after center-reducing they become $-\texttt{y\_mean}/\texttt{y\_std}$, which means nothing either. That is not a problem as long as nothing reads them, which is precisely what the masking below guarantees.

In [ ]:
FOLDER_BACKGROUND = os.path.join(DATA_PATH, "background")

samples = read_annotations(FOLDERS_ANIMALS + [FOLDER_BACKGROUND])

x = load_images(samples)
y = np.stack([encode_localization(boxes) for _, boxes in samples])
groups = np.array([source_photograph(path) for path, _ in samples])

n_present = int((y[:, 0] == 1).sum())
print("images:", len(samples), "| with an object:", n_present, "| without:", len(samples) - n_present)
print("accuracy of a model always answering 'present': {:.1%}".format(n_present / len(samples)))

In [ ]:
(x_train, y_train), (x_val, y_val), (x_test, y_test) = split_dataset(x, y, groups)

# Statistics of the boxes, computed on the training images containing an object
present = y_train[:, 0] == 1
y_mean = y_train[present].mean(axis=0)
y_std = y_train[present].std(axis=0)

for y_set in (y_train, y_val, y_test):
    y_set[:, 1:5] = (y_set[:, 1:5] - y_mean[1:5]) / y_std[1:5]

y_mean_t = torch.tensor(y_mean, dtype=torch.float32, device=device)
y_std_t = torch.tensor(y_std, dtype=torch.float32, device=device)

print("train / val / test:", len(x_train), len(x_val), len(x_test))

A look at the data before training, as always. `draw_localization` draws no box when the presence of the label is zero, so the background crops appear bare.

In [ ]:
with_object = np.where(y_train[:, 0] == 1)[0][:4]
without_object = np.where(y_train[:, 0] == 0)[0][:4]

plt.figure(figsize=(14, 8))
for k, i in enumerate(np.concatenate([with_object, without_object])):
    plt.subplot(2, 4, k + 1)
    draw_localization(x_train[i], y_train[i], "Image " + str(i) + " - {}")

plt.tight_layout()
plt.show()

## A conditional cost function

The presence loss is unchanged: every image says something about it, whether it holds an object or not.

The other two must only count the images that do contain one. The recipe is always the same: compute the loss **image by image** rather than as a single average, multiply by the mask, and divide by the number of images actually kept.

* `reduction='none'` asks a loss of `torch.nn.functional` for one value per image instead of their mean;
* `presence` is of shape $(N, 1)$ and holds $1$ or $0$, so multiplying by it cancels the terms to be ignored;
* dividing by `presence.sum()` gives the average over the images that were kept. The `clamp(min=1)` guards against a batch containing only background, which would otherwise divide by zero.

##### <i style="color:teal">**Todo:** Write the two masked losses</i>

In [ ]:
### TO BE COMPLETED ###

def coord_loss_masked(pred, target, presence):
    """Mean square error, restricted to the images that contain an object."""
    # One value per image, of shape (N, 1)
    per_image = ((pred - target) ** 2).mean(dim=1, keepdim=True)
    return ... ### TO BE COMPLETED ###


def class_loss_masked(pred, target, presence):
    """Cross-entropy, restricted to the images that contain an object."""
    # One value per image, of shape (N, 1)
    per_image = F.cross_entropy(pred, target, reduction='none').unsqueeze(1)
    return ... ### TO BE COMPLETED ###


# The presence loss is the one of Part I: it concerns every image
losses_presence = {'p': presence_loss, 'coord': coord_loss_masked, 'classes': class_loss_masked}

In [ ]:
# %load solutions/VisionCNN/losses_presence.py

##### <i style="color:teal">**Question:** What would happen if these two losses were left unmasked?</i>

**[Solution]**

<!--
The two terms would not misbehave in the same way.

The **coordinate** loss is the harmful one. The target of a background image is $-\texttt{y\_mean}/\texttt{y\_std}$, an arbitrary point that comes from the standardization of a vector of zeros. An unmasked mean square error would push the network to predict that point on 40% of the base, in direct competition with the boxes it has to regress on the rest. The predicted boxes would drift towards that meaningless value.

The **classification** loss is harmless but useless. Its target is a vector of zeros, and the cross-entropy of a soft target is $-\sum_c t_c \log p_c$, which is exactly zero when every $t_c$ is: these images contribute no gradient. They do however enter the average, which divides the whole term by a larger number and therefore shrinks the gradient of the images that do carry information, as if the weight of the classification had been quietly lowered.

In short: masking the coordinates is necessary for correctness, masking the classification only for the loss to keep the meaning we give it.
-->

## Training

Nothing changes in the model: `LocalizationNet` already has its three outputs, and the presence one now has something to predict. Nothing changes in the training loop either, which has been passing the presence to the three losses since Part I.

`run_epoch` restricts `coord_iou` and `classes_accuracy` to the images containing an object, for the same reason that the losses are masked: the IoU of a box that does not exist is not a number worth averaging. `p_accuracy`, on the other hand, is computed over every image, and is now the interesting one — compare it to the rate printed above.

In [ ]:
epochs = 30
batch_size = 15

model = LocalizationNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

train_loader, val_loader = make_loaders(x_train, y_train, x_val, y_val, batch_size)

loss_weights = {'p': 1, 'coord': 5, 'classes': 1}

history = fit_localization(model, train_loader, val_loader, optimizer,
                           losses_presence, loss_weights, epochs)

In [ ]:
plot_results(history)

Predictions on the validation set. The right-hand image shows no box when the network answers that it sees no object, so a false alarm on a background crop, or a missed animal, is immediately visible.

In [ ]:
y_pred = predict_localization(model, x_val)

# A few images with an object, then a few without
indices = list(np.where(y_val[:, 0] == 1)[0][:4]) + list(np.where(y_val[:, 0] == 0)[0][:4])

for i in indices:
    plt.figure(figsize=(10, 5))
    print_data_localization(x_val, y_val, y_pred=y_pred, id=int(i), image_size=IMAGE_SIZE)

plt.show()

## Final evaluation

The `test` set has not been used so far: neither to fit the parameters, nor to compare runs, nor to choose the epoch to keep. Its scores are therefore the only ones that estimate what the model would do on images it has never met, and this is the one and only cell that reads it.

If they are noticeably below the validation scores, the gap measures what was gained by tuning on the validation set — which is exactly why it was kept aside.

In [ ]:
test_loader = DataLoader(make_dataset(x_test, y_test), batch_size=64, shuffle=False)
test_metrics = run_epoch(model, test_loader, losses_presence, loss_weights, optimizer=None)

for name in METRIC_NAMES:
    print("{:20s} val {:.4f}   test {:.4f}".format(name, history['val_' + name][-1], test_metrics[name]))

---
# PART III: Implementation of a simplified version of YOLO

Part II switched some terms of the cost function off, image by image, depending on whether an object was present. We now do the same thing cell by cell: the image is sliced into a grid, and each cell is empty or occupied exactly as a whole image was in the previous part. Everything else follows from there.

In this part, we will try to go further by considering the more complex problem of object detection, _i.e._, the joint localization and classification of all the objects in the image; for that, we will implement a simplified version of YOLO.

The simplification comes from the fact that we will not include all the elements described in [Redmon](https://pjreddie.com/)'s article (for example, the optimizer's choice). One of the main simplifications is that _we will only consider one object per cell_.

<center> <img src="img/YOLO_archi.png" width=500></center>
<caption><center><b>Figure 4</b>: Pipeline of the YOLO algorithm <a href="https://pjreddie.com/media/files/papers/yolo_1.pdf">[Redmon 2016]</a></center></caption>

As a reminder, the idea of YOLO is to slice the image into a grid of cells and perform a prediction of several bounding boxes as well as a classification per cell.

We use here the same dataset : the [Wildlife](https://plmlab.math.cnrs.fr/chevallier-teaching/datasets/wildlife) database.

## Useful functions

Definition of the different variables useful for the following.

In [ ]:
IMAGE_SIZE = 64  # Size of the input images of the network
CELL_PER_DIM = 8 # Number of cells in width and height
BOX_PER_CELL = 1 # Number of objects per cell
NB_CLASSES = 4   # Number of classes of the problem
PIX_PER_CELL = IMAGE_SIZE/CELL_PER_DIM

##### <i style="color:teal">**Question:** What is the size of the $y$-array in the YOLO framework?</i>

**[Solution]**

<!--
$y$ is a vector of size $(S,S,N+5k)$, where:
* $S$ is the grid size, or with the notations introduced above `CELL_PER_DIM`,
* $N$ is the number of classes, _i.e._, `NB_CLASSES`, and
* $k$ is the number of objects detected per cell _i.e._ `BOX_PER_CELL`.

In other words, $y$ is of size (`CELL_PER_DIM`,`CELL_PER_DIM`,`NB_CLASSES`+5*`BOX_PER_CELL`).
-->

### Encoding of the data for the detection problem

The reading of the annotations does not change: `read_annotations` already returns every object of every image, and it is the encoding that differs. Where `encode_localization` kept the largest box and produced a vector of $9$, `encode_yolo` places each object in the cell its centre falls into.

Three conventions of the article are applied here:

* the coordinates of the centre are expressed **relative to the cell** it belongs to, hence between $0$ and $1$ inside that cell;
* the **square root** of the width and of the height is stored rather than the values themselves, so that an error of a few pixels weighs more on a small box than on a large one;
* the class probabilities are shared by the cell and placed at the end of its vector, after the $5B$ entries of the boxes.

With `BOX_PER_CELL = 1`, a cell can hold a single object. When two centres fall in the same cell, the second is lost — the function below counts how often this happens.

##### <i style="color:teal">**Todo:** Complete the encoding function</i>

In [ ]:
### TO BE COMPLETED ###

def encode_yolo(boxes):
    """Turn the list of objects of an image into a (S, S, 5B + C) grid."""
    label = np.zeros((CELL_PER_DIM, CELL_PER_DIM, NB_CLASSES + 5 * BOX_PER_CELL), dtype="f")

    for class_index, cx, cy, width, height in boxes:
        # Indices of the cell the centre of the box falls into.
        # The min guards against a centre sitting exactly on the last edge.
        ind_x = min(int(cx * CELL_PER_DIM), CELL_PER_DIM - 1)
        ind_y = min(int(cy * CELL_PER_DIM), CELL_PER_DIM - 1)

        # YOLO: "The (x, y) coordinates represent the center of the box relative to the
        # bounds of the grid cell" -> coordinates of the centre inside its own cell
        cx_cell = ... ### TO BE COMPLETED ###
        cy_cell = ... ### TO BE COMPLETED ###

        # First free box of that cell; the object is dropped if the cell is already full
        ind_box = 0
        while ind_box < BOX_PER_CELL and label[ind_x, ind_y, 5 * ind_box] == 1:
            ind_box = ind_box + 1
        if ind_box == BOX_PER_CELL:
            continue

        label[ind_x, ind_y, 5 * ind_box] = ...     ### TO BE COMPLETED : presence ###
        label[ind_x, ind_y, 5 * ind_box + 1] = cx_cell
        label[ind_x, ind_y, 5 * ind_box + 2] = cy_cell
        label[ind_x, ind_y, 5 * ind_box + 3] = ... ### TO BE COMPLETED : width ###
        label[ind_x, ind_y, 5 * ind_box + 4] = ... ### TO BE COMPLETED : height ###

        # Class probabilities, shared by the cell and placed after the boxes
        label[ind_x, ind_y, 5 * BOX_PER_CELL:] = ... ### TO BE COMPLETED ###

    return label

In [ ]:
# %load solutions/VisionCNN/encode_yolo.py

In [ ]:
def count_cell_conflicts(samples):
    """Number of images holding two objects whose centres fall in the same cell."""
    conflicts = 0

    for path, boxes in samples:
        cells = [(min(int(cx * CELL_PER_DIM), CELL_PER_DIM - 1),
                  min(int(cy * CELL_PER_DIM), CELL_PER_DIM - 1))
                 for _, cx, cy, _, _ in boxes]
        if len(cells) != len(set(cells)):
            conflicts += 1

    return conflicts

The background crops are left out of this part: with $64$ cells per image and one or two of them occupied, the grid already provides a large majority of empty cells, and the $\lambda_{\text{noobj}}$ term of the loss is there to keep them from dominating.

In [ ]:
samples = read_annotations(FOLDERS_ANIMALS)

x = load_images(samples)
y = np.stack([encode_yolo(boxes) for _, boxes in samples])
groups = np.array([source_photograph(path) for path, _ in samples])

print("images:", len(samples))
print("images losing an object in the grid:", count_cell_conflicts(samples))
print("occupied cells: {:.1%}".format(y[:, :, :, 0].mean()))

x.shape, y.shape

As before, the database is split into training, validation and test data. The coordinates are not center-reduced here: they are already relative to their cell, hence between $0$ and $1$.

In [ ]:
(x_train, y_train), (x_val, y_val), (x_test, y_test) = split_dataset(x, y, groups)

x_train.shape, y_train.shape, x_val.shape, x_test.shape

### Display of data and detection results

In [ ]:
from scipy.special import softmax

##### <i style="color:teal">**Todo:** Complete the following function</i>

Consistent behavior is maintained with the `print_data_localization` function:

1. If only $\texttt{x}$ and $\texttt{y}$ are indicated, an image number is randomly drawn and the $\texttt{y}$ label associated with the image is displayed
2. You can also indicate the $\texttt{id}$ of the image you wish to view.
3. Keep bounding box colors.

The `mode` argument tells the function what it is given: `'gt'` for a ground truth, whose confidence indices and class probabilities are already between $0$ and $1$, and `'pred'` for the raw output of the network, on which the `sigmoid` and the `softmax` still have to be applied.

In [ ]:
### TO BE COMPLETED ###

def print_data_detection(x, y, id=None, image_size=IMAGE_SIZE, mode='gt'):
    if id==None:
        # Random drawing of an image in the database
        num_img = np.random.randint(x.shape[0])
        print(num_img)
    else:
        num_img = id

    img = x[num_img]
    # Work on a copy: the coordinates are modified in place just below
    lab = y[num_img].copy()

    colors = ["royalblue", "limegreen", "purple", "darkorange"] # Different colors for the different classes
    classes = ["Buffalo", "Elephant", "Rhino", "Zebra"]

    boxes = lab[:, :, 1:5]
    for ind_x in range(CELL_PER_DIM):
        for ind_y in range(CELL_PER_DIM):
            box = boxes[ind_x, ind_y]
            box[0] = ... ### TO BE COMPLETED ###
            box[1] = ... ### TO BE COMPLETED ###
            box[2] = box[2]**2 * IMAGE_SIZE
            box[3] = box[3]**2 * IMAGE_SIZE
            boxes[ind_x, ind_y] = box

    # Retrieve all information from bounding boxes
    all_presences = np.reshape(lab[:, :, 0], (CELL_PER_DIM*CELL_PER_DIM))
    all_boxes = np.reshape(lab[:, :, 1:5], (-1, 4))
    all_classes = np.reshape(lab[:, :, 5:9], (-1, 4))

    if mode=='pred':
        all_presences = 1 / (1 + np.exp(-all_presences))
        all_classes = softmax(all_classes, axis=1)

    indices_sorted = np.argsort(-all_presences)

    # Eliminate all bounding boxes whose probability of presence is < threshold
    threshold = 0.35
    all_boxes = ... ### TO BE COMPLETED ###
    all_classes = ... ### TO BE COMPLETED ###
    all_presences = ... ### TO BE COMPLETED ###

    # Image display
    plt.imshow(img)
    for i in range(all_boxes.shape[0]):

        # Determination of the class
        class_id = ... ### TO BE COMPLETED ###
        lab = all_boxes[i]
        # Determination of the extrema of the bounding box
        p_x = [..., ...] ### TO BE COMPLETED ###
        p_y = [..., ...] ### TO BE COMPLETED ###
        # Display the bounding box in the right color
        [...] ### TO BE COMPLETED ###

    plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")

In [ ]:
# %load solutions/VisionCNN/print_data_detection.py

In [ ]:
print_data_detection(x_train, y_train, image_size=IMAGE_SIZE)
plt.show()

## A simplified version of YOLO

The model proposed below is only one possibility among many others. The Redmon article mentions a delicate instability during the training. So, we chose to use an elu (exponential linear unit) activation function.

<center> <img src="img/YOLO.png" width=500></center>
<caption><center><b>Figure 5</b>: YOLO output layer</a></center></caption>

##### <i style="color:teal">**Todo:** Complete the last layer to have the right size output.</i>

The network ends with a fully connected layer, which produces a flat vector: it is up to us to reorganize it into the grid expected by the loss and by the display function. `tensor.view(...)` does exactly that, without moving any data; the $-1$ stands for the batch size, which is not known in advance.

Weight decay is not declared on the layers here, but passed to the optimizer with the `weight_decay` argument, at the very end of the section.

In [ ]:
### TO BE COMPLETED ###

class YOLONet(nn.Module):

    def __init__(self, image_size=IMAGE_SIZE):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding='same'), nn.ELU(),
            nn.Conv2d(32, 32, 3, padding='same'), nn.ELU(),
            nn.Conv2d(32, 32, 3, padding='same'), nn.ELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding='same'), nn.ELU(),
            nn.Conv2d(64, 64, 3, padding='same'), nn.ELU(),
            nn.Conv2d(64, 64, 3, padding='same'), nn.ELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding='same'), nn.ELU(),
            nn.Conv2d(128, 128, 3, padding='same'), nn.ELU(),
            nn.Conv2d(128, 128, 3, padding='same'), nn.ELU(),
            nn.MaxPool2d(2),
        )

        # Three MaxPool2d(2): the spatial size is divided by 8, with 128 channels
        n_features = 128 * (image_size // 8) ** 2

        self.classifier = nn.Sequential(
            nn.Linear(n_features, 512), nn.ELU(),
            nn.Linear(512, 512), nn.ELU(),
            nn.Linear(512, ...), ### TO BE COMPLETED ###
        )

        self.apply(init_he_normal)

    def forward(self, x):
        f = torch.flatten(self.features(x), 1)
        output = self.classifier(f)
        return output.view(...) ### TO BE COMPLETED ###

In [ ]:
# %load solutions/VisionCNN/create_model_YOLO.py

In [ ]:
model = YOLONet().to(device)
print(model)
print("\nNumber of trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

with torch.no_grad():
    print("output:", model(torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)).shape)

### Cost function

We now come to the tricky part of the YOLO implementation: the definition of the cost function to use.

<center> <img src="img/YOLOss.png" style="width:500;height:300px;"></center>
<caption><center> Detail of the loss function defined in the YOLO v1 article</center></caption>

A loss function is called during the training on the outputs of the network, so it works on tensors and must be written with `torch` operations. This is not a formality: only these operations are differentiated by the automatic differentiation engine, which is what makes `loss.backward()` possible. Converting a tensor to `numpy` in the middle of a loss silently breaks that chain.

The operations needed here are few: the elementwise arithmetic, `torch.sigmoid`, `torch.softmax`, and `.sum()`, which adds up all the entries of a tensor.

An essential part of the function is already written: the one that allows the separation of the data of the so-called "empty" cells (the ground truth does not contain a bounding box) from the "non-empty" ones. It relies on *boolean masking*: `y_true[:, 0] >= 1` produces a tensor of booleans, one per cell, and `y_true[mask]` keeps the rows for which the mask is true.

The details of the cost function are shown above: in the article $\lambda_{\text{coord}} = 5$ and $\lambda_{\text{noobj}} = 0.5$.
* The $x_i$, $y_i$, $w_i$, and $h_i$ correspond to the coordinates of a bounding box;
* $C_i$ corresponds to the probability of the presence of an object in the cell (sigmoid function applied to the corresponding output elements); and
* the $p_i(c)$ are the class probabilities (softmax function applied to the corresponding output elements).

##### <i style="color:teal">**Todo:** Complete the following function</i>

In [ ]:
### TO BE COMPLETED ###

# Definition of the YOLO loss function
def YOLOss(lambda_coord, lambda_noobj, batch_size):

    # "Green" part: subpart concerning the confidence index
    # and the class probabilities in the case where a box is present in the cell
    def box_loss(y_pred, y_true):
        return ... ### TO BE COMPLETED ###

    # "Blue" part: subpart concerning the coordinates of the bounding box in the case where a box is present in the cell
    def coord_loss(y_pred, y_true):
        return ... ### TO BE COMPLETED ###

    # "Red" part: subpart concerning the confidence index in case no box is present in the cell
    def nobox_loss(y_pred, y_true):
        return ... ### TO BE COMPLETED ###

    def YOLO_loss(y_pred, y_true):

        # Reshape the tensors from bs x S x S x (5B+C) to (bsxSxS) x (5B+C)
        y_true = y_true.reshape(-1, 9)
        y_pred = y_pred.reshape(-1, 9)

        # Search (in y_true labels) for the cells for which at least the first bounding box is present
        not_empty = y_true[:, 0] >= 1
        empty = ~not_empty

        # Separate the cells of y_true and y_pred with or without bounding box
        y_true_notempty, y_pred_notempty = y_true[not_empty], y_pred[not_empty]
        y_true_empty, y_pred_empty = y_true[empty], y_pred[empty]

        return (box_loss(y_pred_notempty, y_true_notempty)
                + lambda_coord * coord_loss(y_pred_notempty, y_true_notempty)
                + lambda_noobj * nobox_loss(y_pred_empty, y_true_empty)) / batch_size

    # Return a function
    return YOLO_loss

In [ ]:
# %load solutions/VisionCNN/YOLOss.py

### Following the training

The YOLO loss is a sum of squares over a grid: its value says whether the training moves, not whether the detections are any good. Three quantities are followed alongside it, all three restricted to what they can describe.

* the **accuracy of the presence**, over every cell of the grid;
* the **IoU** of the boxes and the **accuracy of the classification**, over the occupied cells only, the same restriction as in Part II.

The IoU is computed between two boxes belonging to the same cell. Their coordinates are the ones stored by `encode_yolo`, a centre relative to the cell and the square root of the size; the offset of the cell is shared by the two boxes, so it cancels out and only the unit of the sizes matters.

In [ ]:
def compute_iou_yolo(y_pred_coord, y_true_coord):
    """IoU between predicted and true boxes belonging to the same cell."""
    pred_box_xy = y_pred_coord[:, 0:2] * PIX_PER_CELL
    true_box_xy = y_true_coord[:, 0:2] * PIX_PER_CELL

    # The stored value is the square root of the size; the clamp guards against
    # the negative values the network is free to output
    pred_box_wh = y_pred_coord[:, 2:4].clamp(min=0) ** 2 * IMAGE_SIZE
    true_box_wh = y_true_coord[:, 2:4] ** 2 * IMAGE_SIZE

    true_mins = true_box_xy - true_box_wh / 2.
    true_maxs = true_box_xy + true_box_wh / 2.
    pred_mins = pred_box_xy - pred_box_wh / 2.
    pred_maxs = pred_box_xy + pred_box_wh / 2.

    intersect_wh = torch.clamp(torch.minimum(pred_maxs, true_maxs)
                               - torch.maximum(pred_mins, true_mins), min=0.)
    intersect_areas = intersect_wh[:, 0] * intersect_wh[:, 1]

    true_areas = true_box_wh[:, 0] * true_box_wh[:, 1]
    pred_areas = pred_box_wh[:, 0] * pred_box_wh[:, 1]

    return intersect_areas / (pred_areas + true_areas - intersect_areas).clamp(min=1e-6)


YOLO_METRIC_NAMES = ['loss', 'p_accuracy', 'coord_iou', 'classes_accuracy']


@torch.no_grad()
def yolo_metrics(y_pred, y_true):
    """Metrics of one batch, and the number of occupied cells they were computed on."""
    y_pred = y_pred.reshape(-1, 9)
    y_true = y_true.reshape(-1, 9)

    occupied = y_true[:, 0] > 0.5
    n_occupied = int(occupied.sum())

    # A cell is predicted as occupied when its score is positive, i.e. sigmoid(score) > 0.5
    metrics = {'p_accuracy': ((y_pred[:, 0] > 0) == occupied).float().mean().item(),
               'coord_iou': 0.0,
               'classes_accuracy': 0.0}

    if n_occupied > 0:
        metrics['coord_iou'] = compute_iou_yolo(y_pred[occupied][:, 1:5],
                                                y_true[occupied][:, 1:5]).mean().item()
        metrics['classes_accuracy'] = (y_pred[occupied][:, 5:9].argmax(1)
                                       == y_true[occupied][:, 5:9].argmax(1)).float().mean().item()

    return metrics, n_occupied

### Training

The loop is the one of Part I with a single loss instead of three, plus one addition: as the training is unstable, we want to keep the model as it was at its best epoch and not as it is at the end.

Nothing does this for us. Saving a model is `torch.save(model.state_dict(), path)`: the `state_dict` is the dictionary holding all the tensors of the model, weights and buffers. Reloading is the symmetrical operation, `model.load_state_dict(torch.load(path))`, on a model of the same architecture. Keeping the best epoch is therefore three lines inside the loop: compare the current validation loss to the best one seen so far, and save when it improves.

> Note that the `state_dict` contains the parameters only, and not the architecture: reloading always requires the class to have been defined first. This is what makes such a file usable from one version to another, and readable at a glance with `list(state_dict.keys())`.

##### <i style="color:teal">**Todo:** Complete the two functions</i>

In [ ]:
### TO BE COMPLETED ###

def run_epoch_yolo(model, loader, criterion, optimizer=None):
    """One pass over `loader`. If `optimizer` is None, the pass is a plain evaluation."""
    is_train = optimizer is not None
    model.train(is_train)

    totals = {name: 0.0 for name in YOLO_METRIC_NAMES}
    n_seen, n_occupied = 0, 0

    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        with torch.set_grad_enabled(is_train):
            y_pred = model(x_batch)
            loss = criterion(y_pred, y_batch)

        if is_train:
            ### TO BE COMPLETED ###
            # Reset the gradients, backpropagate, update the parameters (three lines)
            ...

        metrics, n_batch_occupied = yolo_metrics(y_pred, y_batch)
        n_batch = x_batch.shape[0]
        n_seen += n_batch
        n_occupied += n_batch_occupied

        totals['loss'] += loss.item() * n_batch
        totals['p_accuracy'] += metrics['p_accuracy'] * n_batch
        totals['coord_iou'] += metrics['coord_iou'] * n_batch_occupied
        totals['classes_accuracy'] += metrics['classes_accuracy'] * n_batch_occupied

    counts = {'loss': n_seen, 'p_accuracy': n_seen,
              'coord_iou': n_occupied, 'classes_accuracy': n_occupied}
    return {name: totals[name] / max(counts[name], 1) for name in YOLO_METRIC_NAMES}


def fit_yolo(model, train_loader, val_loader, optimizer, criterion, epochs,
             checkpoint_path="yolo_best.pt"):
    """Train `model`, keeping a copy of the parameters of the best epoch."""
    history = {name: [] for name in YOLO_METRIC_NAMES}
    history.update({'val_' + name: [] for name in YOLO_METRIC_NAMES})
    best_val_loss = float('inf')

    for epoch in range(epochs):
        train_metrics = run_epoch_yolo(model, train_loader, criterion, optimizer=optimizer)
        val_metrics = run_epoch_yolo(model, val_loader, criterion, optimizer=None)

        for name in YOLO_METRIC_NAMES:
            history[name].append(train_metrics[name])
            history['val_' + name].append(val_metrics[name])

        print(f"Epoch {epoch+1}/{epochs}"
              f" - loss: {train_metrics['loss']:.4f} - val_loss: {val_metrics['loss']:.4f}"
              f" - val_presence: {val_metrics['p_accuracy']:.3f}"
              f" - val_IoU: {val_metrics['coord_iou']:.3f}"
              f" - val_class_acc: {val_metrics['classes_accuracy']:.3f}")

        # Save the model each time the validation loss reaches a new minimum
        ### TO BE COMPLETED ###
        [...]

    print(f"\\nBest validation loss: {best_val_loss:.4f}")
    return history

In [ ]:
# %load solutions/VisionCNN/fit_yolo.py

In [ ]:
model = YOLONet().to(device)
batch_size = 18
epochs = 100

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=0)

train_loader, val_loader = make_loaders(x_train, y_train, x_val, y_val, batch_size)

criterion = YOLOss(5, 0.5, batch_size)

history = fit_yolo(model, train_loader, val_loader, optimizer, criterion, epochs,
                   checkpoint_path="yolo_best.pt")

In [ ]:
plt.figure(figsize=(15, 4))

plt.subplot(1, 4, 1) ; plot_training_analysis(history, metric='loss')
plt.subplot(1, 4, 2) ; plot_training_analysis(history, metric='p_accuracy')
plt.subplot(1, 4, 3) ; plot_training_analysis(history, metric='coord_iou')
plt.subplot(1, 4, 4) ; plot_training_analysis(history, metric='classes_accuracy')

plt.tight_layout()
plt.show()

Compare the epoch of the best validation loss with the epochs where the IoU and the accuracy of the classification are at their best: they rarely coincide exactly. The loss is a weighted sum over the whole grid, largely driven by the empty cells, whereas the two other quantities only describe the occupied ones.

The prediction function is the counterpart of `predict_localization`. It returns the raw output of the network: the `sigmoid` and the `softmax` are applied by `print_data_detection` when `mode='pred'`.

In [ ]:
@torch.no_grad()
def predict_yolo(model, x, batch_size=64):
    """Return an array (N, S, S, 9) holding the raw scores of the network."""
    model.eval()
    x_t = torch.tensor(np.transpose(x, (0, 3, 1, 2)), dtype=torch.float32)

    predictions = [model(x_batch.to(device)).cpu()
                   for x_batch in DataLoader(x_t, batch_size=batch_size)]

    return torch.cat(predictions).numpy()

1. Test of the version at the end of the training

In [ ]:
y_pred = predict_yolo(model, x_train)

for i in [2, 3, 8, 15, 18, 24, 26, 32]:
    plt.figure(figsize=(5, 5))
    print_data_detection(x_train, y_pred, id=i, image_size=IMAGE_SIZE, mode='pred')

plt.show()

In [ ]:
y_pred = predict_yolo(model, x_val)

for i in [2, 3, 8, 15, 18, 24, 26, 32]:
    plt.figure(figsize=(5, 5))
    print_data_detection(x_val, y_pred, id=i, image_size=IMAGE_SIZE, mode='pred')

plt.show()

2. Test the _best_ saved version

In [ ]:
model.load_state_dict(torch.load('yolo_best.pt', map_location=device, weights_only=True))
model.eval();

In [ ]:
y_pred = predict_yolo(model, x_train)

for i in [2, 3, 8, 15, 18, 24, 26, 32]:
    plt.figure(figsize=(5, 5))
    print_data_detection(x_train, y_pred, id=i, image_size=IMAGE_SIZE, mode='pred')

plt.show()

In [ ]:
y_pred = predict_yolo(model, x_val)

for i in [2, 3, 8, 15, 18, 24, 26, 32]:
    plt.figure(figsize=(5, 5))
    print_data_detection(x_val, y_pred, id=i, image_size=IMAGE_SIZE, mode='pred')

plt.show()

### Final evaluation

The `test` set of this part has not been used either: the checkpoint was chosen on `val`. These are therefore the scores of the model on images it has never met, and this cell is the only one that reads them.

In [ ]:
test_loader = DataLoader(make_dataset(x_test, y_test), batch_size=64, shuffle=False)
test_metrics = run_epoch_yolo(model, test_loader, criterion, optimizer=None)

for name in YOLO_METRIC_NAMES:
    print("{:20s} val {:.4f}   test {:.4f}".format(name, history['val_' + name][-1], test_metrics[name]))

### Loading weights from an already trained network

As YOLO training is very unstable, it is possible that at the end of this tutorial you will not get very convincing results. To finish this tutorial, you will find next to this notebook the weights of a model trained for a long time, in the file `yolo_pretrained.pt`.

The file is a `state_dict` saved exactly as the loop above does, so loading it only requires the `YOLONet` class to be defined, and nothing else. In particular, no optimizer is involved: the file holds the parameters of the model, and only those.

In [ ]:
PRETRAINED_PATH = "yolo_pretrained.pt"

model = YOLONet().to(device)
model.load_state_dict(torch.load(PRETRAINED_PATH, map_location=device, weights_only=True))
model.eval();

In [ ]:
y_pred = predict_yolo(model, x_train)

for i in [2, 3, 8, 15, 18, 24, 26, 32]:
    plt.figure(figsize=(5, 5))
    print_data_detection(x_train, y_pred, id=i, image_size=IMAGE_SIZE, mode='pred')

plt.show()

In [ ]:
y_pred = predict_yolo(model, x_val)

for i in [2, 3, 8, 15, 18, 24, 26, 32]:
    plt.figure(figsize=(5, 5))
    print_data_detection(x_val, y_pred, id=i, image_size=IMAGE_SIZE, mode='pred')

plt.show()

The results are not perfect, but we are starting to see some good results. As previously, we could limit overfitting by using data augmentation in this training.

We can already see in the few examples below that some of the images are rather well-predicted.

In [ ]:
y_pred = predict_yolo(model, x_val)

for i in [28, 37, 39, 81, 108, 193, 214, 220]:
    plt.figure(figsize=(5, 5))
    print_data_detection(x_val, y_pred, id=i, image_size=IMAGE_SIZE, mode='pred')

plt.show()